In [0]:
# =====================================================
# 1️⃣ Imports & Setup
# =====================================================
# REQUIRED PARAMETERS (jobs inject them; manual runs must supply them):
#   run_id           : numeric run identifier from the orchestrating job
#   table_metadata   : JSON string for the table row (from <catalog>.metadata.tables)
#   table_parameters : JSON string (load_type, primary_key, watermark_column)
#   catalog_name     : Unity Catalog name (e.g. 'banking') — see config/*.json

import json
import time
from pyspark.sql.functions import current_timestamp, col, max as spark_max
from delta.tables import DeltaTable



status = "SUCCESS"
error_message = None
records_read = 0
records_written = 0


# =====================================================
# 2️⃣ Widgets & Fail-Fast Validation
# =====================================================

dbutils.widgets.text(
    "table_metadata",
    "{'table_id': '1', 'table_name': 'customers', 'source_system': 'sqlserver', "
    "'source_schema': 'banking', 'source_table': 'customers', "
    "'source_path': '', 'bronze_schema': 'bronze', "
    "'silver_schema': 'silver', 'active_flag': 'True', "
    "'load_order': '1', 'created_at': '2026-02-18 13:23:37.053711'}"
)

dbutils.widgets.text(
    "table_parameters",
    "{'load_type': 'MERGE', "
    "'primary_key': 'customer_id', "
    "'watermark_column': 'updated_at'}"
)

dbutils.widgets.text("run_id", "")
dbutils.widgets.text("catalog_name", "")

run_id = dbutils.widgets.get("run_id").strip()
if not run_id:
    raise ValueError(
        "run_id is required. Jobs inject it (master job passes {{job.run_id}}); "
        "manual runs must supply a unique numeric value."
    )
if not run_id.isdigit():
    raise ValueError(
        f"run_id must be numeric (pipeline_runs.run_id is BIGINT). Got: {run_id!r}"
    )

catalog_name = dbutils.widgets.get("catalog_name").strip()
if not catalog_name:
    raise ValueError("catalog_name is required (e.g. 'banking' — see config/*.json)")

table_metadata = json.loads(dbutils.widgets.get("table_metadata").replace("'", '"'))
table_parameters = json.loads(dbutils.widgets.get("table_parameters").replace("'", '"'))


table_id = int(table_metadata["table_id"])
table_name = table_metadata["table_name"]
bronze_schema = table_metadata["bronze_schema"]
silver_schema = table_metadata["silver_schema"]

load_type = table_parameters.get("load_type")
primary_key = table_parameters.get("primary_key")
watermark_column = table_parameters.get("watermark_column")

bronze_table = f"{catalog_name}.{bronze_schema}.{table_name}"
silver_table = f"{catalog_name}.{silver_schema}.{table_name}"

print(f"Processing Table: {table_name}")
print(f"Load Type: {load_type}")
print(f"Run ID: {run_id}")

In [0]:
# =====================================================
# 3️⃣ TRY-CATCH BLOCK FOR AUDIT CONTROL (FINAL v4)
# =====================================================
from pyspark.sql import Window as W
from pyspark.sql.functions import row_number

try:
    # ---- [1] Read Bronze ----
    bronze_df = spark.table(bronze_table)

    # ---- [2] Watermark filter (load-type-conditional) ----
    if load_type in ["APPEND", "MERGE"] and watermark_column:
        watermark_df = spark.sql(f"""
            SELECT last_watermark_value
            FROM {catalog_name}.metadata.table_watermarks
            WHERE table_id = {table_id}
        """)
        last_watermark = None
        if watermark_df.count() > 0:
            last_watermark = watermark_df.first()["last_watermark_value"]
        if last_watermark:
            if load_type == "MERGE":
                # Idempotent with the freshness guard at [5]: boundary rows are
                # re-read and updated to identical values (never skipped forever).
                bronze_df = bronze_df.filter(col(watermark_column) >= last_watermark)
            else:  # APPEND keeps strict '>': an accidental APPEND run can never
                   # re-append a row the target already holds.
                bronze_df = bronze_df.filter(col(watermark_column) > last_watermark)

    # ---- [3] Batch deduplication — BEFORE audit stamping ----
    if primary_key:
        order_cols = []
        if watermark_column:
            order_cols.append(col(watermark_column).desc_nulls_last())
        elif "created_at" in bronze_df.columns:
            order_cols.append(col("created_at").desc_nulls_last())   # branches
        order_cols.append(col("insert_timestamp").desc_nulls_last())  # Bronze-native
        bronze_df = (bronze_df
            .withColumn("_rn", row_number().over(
                W.partitionBy(primary_key).orderBy(*order_cols)))
            .filter("_rn = 1")
            .drop("_rn"))

    records_read = bronze_df.count()
    print("Records to process:", records_read)

    if records_read > 0:   # empty-batch guard: never write/overwrite 0 rows

        # ---- [4] Audit columns (AFTER dedup — placement as original) ----
        bronze_df = (bronze_df
            .withColumn("insert_timestamp", current_timestamp())
            .withColumn("update_timestamp", current_timestamp()))

        spark.sql(f"create schema if not exists {catalog_name}.{silver_schema}")

        if not spark.catalog.tableExists(silver_table):
            (bronze_df.write.format("delta").mode("overwrite").saveAsTable(silver_table))
        else:
            if load_type == "FULL":
                (bronze_df.write.format("delta").mode("overwrite")
                 .option("overwriteSchema", "true").saveAsTable(silver_table))
            elif load_type == "APPEND":
                (bronze_df.write.format("delta").mode("append").saveAsTable(silver_table))
            elif load_type == "MERGE":
                if not primary_key:
                    raise ValueError("Primary key required for MERGE.")
                delta_table = DeltaTable.forName(spark, silver_table)
                merge_condition = f"t.{primary_key} = s.{primary_key}"
                (delta_table.alias("t")
                    .merge(bronze_df.alias("s"), merge_condition)
                    .whenMatchedUpdateAll(
                        condition=(f"s.{watermark_column} >= t.{watermark_column}"
                                   if watermark_column else None))
                    .whenNotMatchedInsertAll()
                    .execute())
            else:
                raise ValueError("Unsupported load_type")

        # ---- [5] ACTUAL rows written (Delta history metrics — not attempted) ----
        hist = spark.sql(f"DESCRIBE HISTORY {silver_table} LIMIT 1").collect()[0]
        metrics = hist["operationMetrics"] or {}
        if hist["operation"] == "MERGE":
            records_written = (int(float(metrics.get("numTargetRowsInserted", 0)))
                               + int(float(metrics.get("numTargetRowsUpdated", 0))))
        else:
            records_written = int(float(metrics.get("numOutputRows", records_read)))
        print("Records actually written:", records_written)

    else:
        print("Empty batch: write skipped; watermark unchanged.")

    # ---- [6] Watermark update (unchanged; skipped when max is None) ----
    if load_type in ["APPEND", "MERGE"] and watermark_column:
        max_value = bronze_df.agg(spark_max(col(watermark_column))).collect()[0][0]
        if max_value:
            spark.sql(f"""
                MERGE INTO {catalog_name}.metadata.table_watermarks t
                USING (SELECT {table_id} AS table_id) s
                ON t.table_id = s.table_id
                WHEN MATCHED THEN UPDATE SET
                    last_watermark_value = '{max_value}',
                    last_updated_at = current_timestamp()
                WHEN NOT MATCHED THEN
                    INSERT (table_id, last_watermark_value, last_updated_at)
                    VALUES ({table_id}, '{max_value}', current_timestamp())
            """)

    print("Silver Load Completed Successfully.")

except Exception as e:
    status = "FAILED"
    error_message = str(e)
    print("Error Occurred:", error_message)
    raise

finally:
    end_time = spark.sql("SELECT current_timestamp()").collect()[0][0]
    spark.sql(f"""
        UPDATE {catalog_name}.metadata.pipeline_runs
        SET end_time = TIMESTAMP('{end_time}'),
            status = '{status}',
            number_of_records = {records_read},
            error_message = {'NULL' if not error_message else "'" + error_message.replace("'", "") + "'"}
        WHERE table_id = {table_id} and run_id = {run_id}
    """)
    print("Audit record inserted.")